# NYC yellow taxi trip duration

This notebook builds a complete first version of the project: prepare January 2025 taxi data, train a random forest, record the experiment in MLflow, save the model, and test predictions through the API.

The project source code lives in `src/` and `app/`. The small tests below use made-up rows, so they run quickly without downloading the full taxi dataset.

Before running the data cells, make sure the selected notebook kernel is the project's Python 3.13 environment.

In [2]:
from pathlib import Path
import sys

import pandas as pd
import requests
from fastapi.testclient import TestClient
from IPython.display import display

# use the repository root for consistent data and model paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook with the project root as the working directory")

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
DATA_PATH = PROJECT_ROOT / "data" / "yellow_tripdata_2025-01.parquet"
MODEL_PATH = PROJECT_ROOT / "artifacts" / "taxi_duration_pipeline.joblib"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import FEATURE_COLUMNS, TARGET_COLUMN, build_model, prepare_training_data
from src.train import train_model

display(pd.DataFrame({"project root": [str(PROJECT_ROOT)], "data file": [str(DATA_PATH)]}))

,project root,data file
0,c:\Users\shiar\Desktop\neuefischer\model-deplo...,c:\Users\shiar\Desktop\neuefischer\model-deplo...


## 1. Download and inspect the dataset

The raw file is several gigabytes when expanded in memory. The download cell saves it once in `data/`; do not add the data file to Git. Later, training uses a reproducible sample to keep a first run manageable.

In [3]:
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    response = requests.get(DATA_URL, stream=True, timeout=(20, 180))
    response.raise_for_status()
    with DATA_PATH.open("wb") as data_file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                data_file.write(chunk)
    print(f"downloaded dataset to {DATA_PATH}")
else:
    print(f"using existing dataset at {DATA_PATH}")

using existing dataset at c:\Users\shiar\Desktop\neuefischer\model-deployment-project\amin\nf-mle-model-deployment-project\data\yellow_tripdata_2025-01.parquet


In [4]:
raw_data = pd.read_parquet(DATA_PATH)
print(f"rows: {len(raw_data):,}")
print(f"columns: {len(raw_data.columns)}")
display(raw_data.head())
display(raw_data.isna().sum().sort_values(ascending=False).head(10).to_frame("missing values"))

rows: 3,475,226
columns: 20


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


,missing values
passenger_count,540149
congestion_surcharge,540149
store_and_fwd_flag,540149
RatecodeID,540149
Airport_fee,540149
tpep_dropoff_datetime,0
VendorID,0
tpep_pickup_datetime,0
DOLocationID,0
payment_type,0


## 2. Clean trips and define the prediction target

The target is the elapsed time between pickup and drop-off, measured in minutes. The model only receives distance, passenger count, pickup hour and weekday, and pickup and drop-off zone IDs. We remove impossible durations and clearly invalid distances or passenger counts; these are simple baseline rules to review after inspecting the real data.

In [5]:
prepared_data = prepare_training_data(raw_data)
print(f"usable rows: {len(prepared_data):,} of {len(raw_data):,}")
display(prepared_data.describe().T)

usable rows: 2,855,848 of 3,475,226


,count,mean,std,min,25%,50%,75%,max
trip_distance,2855848.0,3.204837,4.337262,0.1,1.0,1.63,3.030000,97.580000
passenger_count,2855848.0,1.307212,0.741620,1.0,1.0,1.00,1.000000,6.000000
pickup_hour,2855848.0,14.391885,5.531035,0.0,11.0,15.00,19.000000,23.000000
pickup_weekday,2855848.0,3.019945,1.837581,0.0,2.0,3.00,4.000000,6.000000
PULocationID,2855848.0,166.946059,62.886235,1.0,132.0,162.00,234.000000,265.000000
DOLocationID,2855848.0,166.014396,68.902004,1.0,116.0,163.00,234.000000,265.000000
duration_minutes,2855848.0,14.655289,11.734185,1.0,7.1,11.40,18.183333,179.966667


## 3. Train a baseline and record it in MLflow

Training samples up to 50,000 cleaned rows, then holds out 20 percent for validation. This is deliberately a modest first run. The model, settings and validation score are recorded in local MLflow storage, and the fitted preprocessing-plus-model pipeline is saved for the API.

In [6]:
training_result = train_model(raw_data, sample_size=50_000, random_state=42)
display(pd.Series(training_result, name="training result"))

2026/10/01 13:37:46 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\shiar\Desktop\neuefischer\model-deployment-project\amin\nf-mle-model-deployment-project
2026/10/01 13:37:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/01 13:37:46 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in c:\Users\shiar\Desktop\neuefischer\model-deployment-project\amin\nf-mle-model-deployment-project
2026/10/01 13:37:46 INFO mlflow.utils.environment: Detected uv project at c:\Users\shiar\Desktop\neuefischer\model-deployment-project\amin\nf-mle-model-deployment-project. Attempting to export re

rmse_minutes                                                5.265375
training_rows                                                  40000
validation_rows                                                10000
run_id                              e49cac4d828943ec8ec87a0ae9319ce5
model_path         C:\Users\shiar\Desktop\neuefischer\model-deplo...
Name: training result, dtype: object

Open a separate terminal from the project root to inspect recorded runs:

```powershell
uv run mlflow ui --backend-store-uri ./mlruns/mlflow.db --port 5000
```

Then open `http://127.0.0.1:5000` in a browser. The reported RMSE is in minutes; compare it with a simple baseline before deciding whether more tuning is worthwhile.

## 4. Run small tests before using real data

These checks cover invalid-row removal, feature creation, a tiny model fit, input validation and a successful API prediction. They use synthetic records and should finish in a few seconds.

In [8]:
from datetime import timedelta

import numpy as np

# generate compact trips for repeatable checks without downloading taxi data

def make_test_trips(row_count=60):
    pickup_times = pd.date_range("2025-01-01 08:00:00", periods=row_count, freq="h")
    return pd.DataFrame(
        {
            "tpep_pickup_datetime": pickup_times,
            "tpep_dropoff_datetime": pickup_times + pd.to_timedelta(10, unit="m"),
            "trip_distance": np.linspace(1, 12, row_count),
            "passenger_count": np.ones(row_count, dtype=int),
            "PULocationID": np.where(np.arange(row_count) % 2, 161, 162),
            "DOLocationID": np.where(np.arange(row_count) % 3, 236, 237),
        }
    )


def test_cleaning_rejects_invalid_trips():
    raw = make_test_trips(4)
    raw.loc[1, "tpep_dropoff_datetime"] = raw.loc[1, "tpep_pickup_datetime"] - timedelta(minutes=2)
    raw.loc[2, "trip_distance"] = 0
    raw.loc[3, "tpep_pickup_datetime"] = pd.NaT
    cleaned = prepare_training_data(raw)
    assert len(cleaned) == 1
    assert cleaned.loc[0, TARGET_COLUMN] == 10
    assert cleaned.loc[0, "pickup_hour"] == 8


def test_model_learns_from_small_sample():
    cleaned = prepare_training_data(make_test_trips())
    model = build_model()
    model.fit(cleaned[FEATURE_COLUMNS], cleaned[TARGET_COLUMN])
    predictions = model.predict(cleaned[FEATURE_COLUMNS].head(5))
    assert len(predictions) == 5
    assert np.isfinite(predictions).all()


def test_api_validates_and_predicts():
    from fastapi.testclient import TestClient
    import app.main as api

    original_model_path = api.MODEL_PATH
    api.MODEL_PATH = PROJECT_ROOT / "artifacts" / "missing-test-model.joblib"
    api.MODEL = None
    client = TestClient(api.app)
    missing_model_response = client.post(
        "/predict",
        json={
            "trip_distance": 2.5,
            "passenger_count": 1,
            "pickup_hour": 9,
            "pickup_weekday": 2,
            "PULocationID": 161,
            "DOLocationID": 236,
        },
    )
    assert missing_model_response.status_code == 503

    cleaned = prepare_training_data(make_test_trips())
    test_model = build_model().fit(cleaned[FEATURE_COLUMNS], cleaned[TARGET_COLUMN])
    api.MODEL = test_model
    valid_response = client.post(
        "/predict",
        json={
            "trip_distance": 2.5,
            "passenger_count": 1,
            "pickup_hour": 9,
            "pickup_weekday": 2,
            "PULocationID": 161,
            "DOLocationID": 236,
        },
    )
    invalid_response = client.post(
        "/predict",
        json={
            "trip_distance": 2.5,
            "passenger_count": 1,
            "pickup_hour": 25,
            "pickup_weekday": 2,
            "PULocationID": 161,
            "DOLocationID": 236,
        },
    )
    assert valid_response.status_code == 200
    assert valid_response.json()["predicted_duration_minutes"] > 0
    assert invalid_response.status_code == 422
    api.MODEL = None
    api.MODEL_PATH = original_model_path


test_cleaning_rejects_invalid_trips()
test_model_learns_from_small_sample()
test_api_validates_and_predicts()
print("passed: cleaning, model fit, missing-model response, prediction and request validation")

passed: cleaning, model fit, missing-model response, prediction and request validation


## 5. Check the trained model through the API

After the training cell succeeds, this request loads the saved pipeline and sends one real validation row through the same input checks used by the service.

In [9]:
from sklearn.model_selection import train_test_split
import app.main as api

clean_features = prepared_data[FEATURE_COLUMNS]
clean_target = prepared_data[TARGET_COLUMN]
_, validation_features, _, _ = train_test_split(
    clean_features,
    clean_target,
    test_size=0.2,
    random_state=42,
)

api.MODEL = None
api.MODEL_PATH = MODEL_PATH
client = TestClient(api.app)
request_payload = validation_features.iloc[0].to_dict()
request_payload = {key: value.item() if hasattr(value, "item") else value for key, value in request_payload.items()}
response = client.post("/predict", json=request_payload)
response.raise_for_status()
print(response.json())

{'predicted_duration_minutes': 14.829725236048247}


To run the service as a separate local process, open a terminal at the project root after training:

```powershell
uv run uvicorn app.main:app --reload --port 8000
```

Open `http://127.0.0.1:8000/docs` to try the request form, or send this example from PowerShell:

```powershell
$body = @{
    trip_distance = 2.5
    passenger_count = 1
    pickup_hour = 9
    pickup_weekday = 2
    PULocationID = 161
    DOLocationID = 236
} | ConvertTo-Json

Invoke-RestMethod -Uri "http://127.0.0.1:8000/predict" -Method Post -ContentType "application/json" -Body $body
```

The API returns a predicted duration in minutes. Record the actual validation RMSE and any data-cleaning limitations in the project README after running the full-data cells.